# Ingesta de Datos de Fraude en Hadoop (HDFS) con PySpark

**Objetivo:** Descargar el dataset de transacciones financieras nigerianas desde HuggingFace,  
cargarlo con PySpark, aplicar transformaciones básicas y persistirlo en HDFS.

**Dataset:** `electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset`  
**HDFS paths:**  
- Raw  → `hdfs://hadoop:9000/data/raw/transactions.parquet`  
- Processed → `hdfs://hadoop:9000/data/processed/transactions_clean.parquet`  
- Results → `hdfs://hadoop:9000/data/fraud-results/`

> Este notebook se ejecuta desde **JupyterLab** (`ml-env`, puerto 8888).  
> El contenedor Hadoop debe estar levantado: `docker compose -f infrastructure/hadoop/docker-compose.yml up -d`

## 1. Instalación de dependencias

In [ ]:
# Solo necesario la primera vez en el entorno ml-env
import subprocess, sys

pkgs = ["pyspark==3.5.3", "pyarrow==16.1.0", "datasets==2.20.0", "huggingface_hub==0.23.4"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=True)

print("Dependencias instaladas correctamente.")

## 2. Descargar dataset desde HuggingFace

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Descargando dataset de HuggingFace...")
dataset = load_dataset(
    "electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset",
    trust_remote_code=True
)

# Convertir a pandas para inspección rápida
df_raw = dataset["train"].to_pandas()
print(f"Filas: {len(df_raw):,}  |  Columnas: {df_raw.columns.tolist()}")
df_raw.head(3)

## 3. Inicializar SparkSession conectada a HDFS

In [ ]:
import os
from pyspark.sql import SparkSession

# HDFS hostname resolvible dentro de la red shared-ml-network
HDFS_URI = "hdfs://hadoop:9000"

spark = (
    SparkSession.builder
    .appName("FraudDetection-Ingesta")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", HDFS_URI)
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} iniciado. HDFS: {HDFS_URI}")

## 4. Cargar pandas → Spark DataFrame y guardar en HDFS (raw)

In [ ]:
from pyspark.sql.types import *

# Convertir pandas a Spark
sdf_raw = spark.createDataFrame(df_raw)
print(f"Spark DataFrame: {sdf_raw.count():,} filas, {len(sdf_raw.columns)} columnas")
sdf_raw.printSchema()

# Guardar raw en HDFS como Parquet
RAW_PATH = f"{HDFS_URI}/data/raw/transactions.parquet"
sdf_raw.write.mode("overwrite").parquet(RAW_PATH)
print(f"[OK] Dataset raw guardado en {RAW_PATH}")

## 5. Transformaciones y limpieza de datos

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan, count

# Leer desde HDFS (buenas prácticas: trabajar desde el origen persistido)
sdf = spark.read.parquet(RAW_PATH)

print("=== Nulos por columna ===")
sdf.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in sdf.columns]).show()

# Eliminar filas con nulos en columnas críticas
critical_cols = ["amount", "oldbalanceOrg", "newbalanceOrig"]
sdf_clean = sdf.dropna(subset=[c for c in critical_cols if c in sdf.columns])

# Normalizar nombre columna tipo de transacción si existe
if "type" in sdf_clean.columns:
    sdf_clean = sdf_clean.withColumn("type", F.upper(F.trim(col("type"))))

# Filtrar importes negativos (datos corruptos)
if "amount" in sdf_clean.columns:
    sdf_clean = sdf_clean.filter(col("amount") > 0)

print(f"Filas tras limpieza: {sdf_clean.count():,}")
sdf_clean.show(5, truncate=False)

## 6. Feature engineering para el modelo de fraude

In [ ]:
from pyspark.sql.functions import lit

# One-hot encoding manual del tipo de transacción
tipos = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]

sdf_features = sdf_clean
for t in tipos:
    col_name = f"type_{t}"
    if "type" in sdf_features.columns:
        sdf_features = sdf_features.withColumn(
            col_name,
            when(col("type") == t, 1.0).otherwise(0.0)
        )
    else:
        sdf_features = sdf_features.withColumn(col_name, lit(0.0))

# Renombrar columnas al formato esperado por el modelo (igual que el API)
rename_map = {
    "oldbalanceOrg":  "old_balance_orig",
    "newbalanceOrig": "new_balance_orig",
    "oldbalanceDest": "old_balance_dest",
    "newbalanceDest": "new_balance_dest",
    "isFraud":        "label"
}
for old_name, new_name in rename_map.items():
    if old_name in sdf_features.columns:
        sdf_features = sdf_features.withColumnRenamed(old_name, new_name)

# Columnas de grafos (PageRank, comunidades) — placeholders para Neo4j
for graph_col in ["orig_out_degree", "orig_pagerank", "orig_community",
                  "dest_in_degree", "dest_pagerank", "dest_community"]:
    if graph_col not in sdf_features.columns:
        sdf_features = sdf_features.withColumn(graph_col, lit(0.0))

print(f"Features preparadas: {len(sdf_features.columns)} columnas")
sdf_features.printSchema()

## 7. Guardar datos procesados en HDFS

In [ ]:
PROCESSED_PATH = f"{HDFS_URI}/data/processed/transactions_clean.parquet"

sdf_features.write.mode("overwrite").parquet(PROCESSED_PATH)
print(f"[OK] Datos procesados guardados en {PROCESSED_PATH}")

# Verificar en HDFS
import subprocess
result = subprocess.run(
    ["hdfs", "dfs", "-du", "-s", "-h", "/data/"],
    capture_output=True, text=True
)
print("Uso de espacio HDFS /data/:")
print(result.stdout or result.stderr)

## 8. Llamar al API de fraude con muestras del dataset

In [ ]:
import requests
import json

API_URL = "http://ai-service:8000/predict"  # nombre del servicio en shared-ml-network

# Seleccionar 5 muestras del dataset procesado
sample_rows = sdf_features.select(
    "amount", "old_balance_orig", "new_balance_orig",
    "old_balance_dest", "new_balance_dest",
    "orig_out_degree", "orig_pagerank", "orig_community",
    "dest_in_degree", "dest_pagerank", "dest_community",
    "type_CASH_IN", "type_CASH_OUT", "type_DEBIT",
    "type_PAYMENT", "type_TRANSFER"
).limit(5).toPandas()

results = []
for _, row in sample_rows.iterrows():
    payload = row.to_dict()
    try:
        resp = requests.post(API_URL, json=payload, timeout=5)
        resp.raise_for_status()
        results.append(resp.json())
    except Exception as e:
        results.append({"error": str(e)})

print("Resultados del API para 5 transacciones:")
for i, r in enumerate(results):
    print(f"  [{i+1}] {r}")

## 9. Guardar predicciones en HDFS

In [ ]:
import requests

# Broadcast de la URL para usar en UDF distribuida
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, StructType, StructField

API_URL_LOCAL = "http://ai-service:8000/predict"

def predict_fraud(amount, old_bal_orig, new_bal_orig, old_bal_dest, new_bal_dest,
                  out_deg, pr_orig, comm_orig, in_deg, pr_dest, comm_dest,
                  ci, co, deb, pay, tr):
    payload = {
        "amount": float(amount or 0),
        "old_balance_orig": float(old_bal_orig or 0),
        "new_balance_orig": float(new_bal_orig or 0),
        "old_balance_dest": float(old_bal_dest or 0),
        "new_balance_dest": float(new_bal_dest or 0),
        "orig_out_degree": float(out_deg or 0),
        "orig_pagerank": float(pr_orig or 0),
        "orig_community": float(comm_orig or 0),
        "dest_in_degree": float(in_deg or 0),
        "dest_pagerank": float(pr_dest or 0),
        "dest_community": float(comm_dest or 0),
        "type_CASH_IN": float(ci or 0),
        "type_CASH_OUT": float(co or 0),
        "type_DEBIT": float(deb or 0),
        "type_PAYMENT": float(pay or 0),
        "type_TRANSFER": float(tr or 0),
    }
    try:
        r = requests.post(API_URL_LOCAL, json=payload, timeout=3)
        return float(r.json().get("fraud_probability", -1))
    except:
        return -1.0

predict_udf = udf(predict_fraud, DoubleType())

# Aplicar al primer batch (1000 filas para no saturar)
sdf_batch = sdf_features.limit(1000)
sdf_with_pred = sdf_batch.withColumn(
    "fraud_probability",
    predict_udf(
        "amount", "old_balance_orig", "new_balance_orig",
        "old_balance_dest", "new_balance_dest",
        "orig_out_degree", "orig_pagerank", "orig_community",
        "dest_in_degree", "dest_pagerank", "dest_community",
        "type_CASH_IN", "type_CASH_OUT", "type_DEBIT",
        "type_PAYMENT", "type_TRANSFER"
    )
)

RESULTS_PATH = f"{HDFS_URI}/data/fraud-results/batch_predictions.parquet"
sdf_with_pred.write.mode("overwrite").parquet(RESULTS_PATH)
print(f"[OK] Predicciones guardadas en {RESULTS_PATH}")

# Resumen de fraudes detectados
fraud_count = sdf_with_pred.filter(col("fraud_probability") >= 0.15).count()
total = sdf_with_pred.count()
print(f"Fraudes detectados: {fraud_count}/{total} ({100*fraud_count/total:.1f}%)")

## 10. Verificación final del estado de HDFS

In [ ]:
# Listar estructura de /data en HDFS
print("=== Estructura HDFS /data/ ===")
for path in ["/data/raw", "/data/processed", "/data/fraud-results"]:
    files = spark._jvm.org.apache.hadoop.fs.FileSystem \
        .get(spark._jsc.hadoopConfiguration()) \
        .listStatus(spark._jvm.org.apache.hadoop.fs.Path(path))
    print(f"\n{path}/")
    for f in files:
        size_mb = f.getLen() / 1024 / 1024
        print(f"  {f.getPath().getName()}  ({size_mb:.2f} MB)")

spark.stop()
print("\nSparkSession cerrada. Ingesta completada.")